# ⚙️ Phase 4: Feature Engineering
**Project:** AI-Powered Customer Intelligence & Revenue Optimization Platform  
**Analyst:** Naman Roy  

---
### Objective
Transform cleaned transaction tables into rich analytical features spanning time, delivery SLAs, basket metrics, and customer-level intelligence using persistent `customer_unique_id`.


In [ ]:
import os
import pandas as pd
import numpy as np

clean_dir = '../../02_Data/02_Cleaned_Data'
proc_dir = '../../02_Data/03_Processed_Data'
os.makedirs(proc_dir, exist_ok=True)

orders = pd.read_csv(os.path.join(clean_dir, 'orders_clean.csv'), parse_dates=[
    'order_purchase_timestamp', 'order_delivered_customer_date', 'order_estimated_delivery_date'
])
customers = pd.read_csv(os.path.join(clean_dir, 'customers_clean.csv'))
payments = pd.read_csv(os.path.join(clean_dir, 'payments_clean.csv'))
order_items = pd.read_csv(os.path.join(clean_dir, 'order_items_clean.csv'))
reviews = pd.read_csv(os.path.join(clean_dir, 'reviews_clean.csv'))

print("✅ Clean datasets loaded.")


### 1. Delivery & Temporal Features

In [ ]:
delivered = orders[orders['order_status'] == 'delivered'].copy()

# Temporal
delivered['order_year'] = delivered['order_purchase_timestamp'].dt.year
delivered['order_month'] = delivered['order_purchase_timestamp'].dt.month
delivered['order_quarter'] = delivered['order_purchase_timestamp'].dt.quarter
delivered['order_dow'] = delivered['order_purchase_timestamp'].dt.day_name()
delivered['is_weekend'] = delivered['order_purchase_timestamp'].dt.dayofweek >= 5

# Delivery SLAs
delivered['delivery_days'] = (delivered['order_delivered_customer_date'] - delivered['order_purchase_timestamp']).dt.days
delivered['delivery_delay'] = (delivered['order_delivered_customer_date'] - delivered['order_estimated_delivery_date']).dt.days
delivered['is_late'] = (delivered['delivery_delay'] > 0).astype(int)

print(f"Delivered Orders: {len(delivered):,}")
print(f"Late Deliveries : {delivered['is_late'].sum():,} ({delivered['is_late'].mean()*100:.1f}%)")


### 2. Basket & Order Revenue Features

In [ ]:
order_rev = payments.groupby('order_id').agg(
    order_value=('payment_value', 'sum'),
    num_payments=('payment_sequential', 'max')
).reset_index()

basket = order_items.groupby('order_id').agg(
    basket_size=('order_item_id', 'max'),
    avg_item_price=('price', 'mean'),
    total_freight=('freight_value', 'sum')
).reset_index()

print(f"Average Order Value: ${order_rev['order_value'].mean():.2f}")
print(f"Average Basket Size: {basket['basket_size'].mean():.2f} items")


### 3. Customer-Level Aggregations (`customer_unique_id`)

In [ ]:
ord_cust = delivered.merge(customers[['customer_id', 'customer_unique_id', 'customer_state']], on='customer_id')
ord_cust = ord_cust.merge(order_rev, on='order_id').merge(basket, on='order_id')

rev_scores = reviews.groupby('order_id')['review_score'].mean().reset_index()
ord_cust = ord_cust.merge(rev_scores, on='order_id', how='left')

ref_date = delivered['order_purchase_timestamp'].max() + pd.Timedelta(days=1)

customer_features = ord_cust.groupby('customer_unique_id').agg(
    total_orders=('order_id', 'nunique'),
    total_spend=('order_value', 'sum'),
    avg_order_value=('order_value', 'mean'),
    total_items=('basket_size', 'sum'),
    avg_freight=('total_freight', 'mean'),
    avg_review_score=('review_score', 'mean'),
    avg_delivery_delay=('delivery_delay', 'mean'),
    late_order_ratio=('is_late', 'mean'),
    first_purchase=('order_purchase_timestamp', 'min'),
    last_purchase=('order_purchase_timestamp', 'max'),
    state=('customer_state', 'first')
).reset_index()

customer_features['tenure_days'] = (ref_date - customer_features['first_purchase']).dt.days
customer_features['recency_days'] = (ref_date - customer_features['last_purchase']).dt.days
customer_features['is_repeat_buyer'] = (customer_features['total_orders'] > 1).astype(int)
customer_features['freight_ratio'] = customer_features['avg_freight'] / (customer_features['avg_order_value'] + 1e-5)

delivered.to_csv(os.path.join(proc_dir, 'orders_featured.csv'), index=False)
order_rev.to_csv(os.path.join(proc_dir, 'order_revenue.csv'), index=False)
basket.to_csv(os.path.join(proc_dir, 'basket_features.csv'), index=False)
customer_features.to_csv(os.path.join(proc_dir, 'customer_features.csv'), index=False)

print(f"✅ Generated customer features for {len(customer_features):,} unique individuals.")
